## 14_scaling_by_country
  Scaling exponent per country (CZ, PL, HU, SK), with and without capitals, for GDP, employment
  and three POI indicators. Output: results/scaling_by_country/*.csv.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make the project root importable (notebook runs from a subfolder or root)
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import NUTS3_MASTER_TABLE_PATH, RESULTS_DIR

# ---- Config block: change names/paths only here ----
ID_COL = "nuts3_id"
YEAR_COL = "year"
COUNTRY_COL = "country_code"
POP_COL = "population"
Y_COLS = [
    "gdp_total", "employment", "active_enterprises",
    "poi_count_university", "poi_count_office", "poi_count_library",
]  # dependent variables
CAPITAL_IDS = {"HU110", "CZ010", "SK010", "PL911"}  # capital NUTS3 regions
OUT_DIR = RESULTS_DIR / "scaling_by_country"
# ----------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load master table, drop geometry (not needed here)
master = pd.read_parquet(NUTS3_MASTER_TABLE_PATH)
master = master.drop(columns=["geometry"], errors="ignore")

# Print only relative paths
print("input:", NUTS3_MASTER_TABLE_PATH.relative_to(ROOT))
print("output dir:", OUT_DIR.relative_to(ROOT))
print("shape:", master.shape)
print("years:", sorted(master[YEAR_COL].unique()))
print("regions:", master[ID_COL].nunique())
print(master.groupby(COUNTRY_COL)[ID_COL].nunique())

# Missing values in the columns we will use
print(master[[POP_COL] + Y_COLS].isna().groupby(master[YEAR_COL]).sum())

In [ ]:
import numpy as np
from scipy import stats

from src.scaling_stats import fit_scaling_law

# ---- Config block ----
FIT_YEAR = 2023
ALL_GROUP = "V4"                      # label for the pooled sample
OUT_FILE = OUT_DIR / "beta_by_country_2023.csv"
# ----------------------

def fit_with_hc3(data, y_col, n_col):
    """OLS fit via fit_scaling_law, plus HC3 standard error and CI."""
    res = fit_scaling_law(data, y_col=y_col, n_col=n_col)
    model = res.pop("model")
    se_hc3 = float(model.HC3_se[n_col])
    t_crit = stats.t.ppf(0.975, model.df_resid)
    res["beta_se_hc3"] = se_hc3
    res["beta_ci_low_hc3"] = res["beta"] - t_crit * se_hc3
    res["beta_ci_high_hc3"] = res["beta"] + t_crit * se_hc3
    return res

if OUT_FILE.exists():
    print("skip, already exists:", OUT_FILE.relative_to(ROOT))
    beta_table = pd.read_csv(OUT_FILE)
else:
    data_year = master[master[YEAR_COL] == FIT_YEAR]
    groups = {c: data_year[data_year[COUNTRY_COL] == c]
              for c in sorted(data_year[COUNTRY_COL].unique())}
    groups[ALL_GROUP] = data_year

    rows = []
    for group, gdf in groups.items():
        samples = {
            "full": gdf,
            "no_capitals": gdf[~gdf[ID_COL].isin(CAPITAL_IDS)],
        }
        for sample, sdf in samples.items():
            for y_col in Y_COLS:
                try:
                    row = fit_with_hc3(sdf, y_col, POP_COL)
                except Exception as e:
                    # Print only the exception class name (no paths)
                    print("failed:", group, sample, y_col, type(e).__name__)
                    continue
                row.update({"group": group, "sample": sample})
                rows.append(row)

    beta_table = pd.DataFrame(rows)
    beta_table.to_csv(OUT_FILE, index=False)
    print("saved:", OUT_FILE.relative_to(ROOT))

show_cols = ["group", "sample", "y_variable", "beta", "beta_se",
             "beta_se_hc3", "r_squared", "n_regions", "breusch_pagan_pvalue"]
print(beta_table[show_cols].round(3).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

from src.config import FIGURES_DIR

# ---- Config block ----
FIG_DIR = FIGURES_DIR / "scaling_by_country"
COUNTRY_COLORS = {"CZ": "tab:blue", "HU": "tab:green", "PL": "tab:orange", "SK": "tab:purple"}
FIG_SIZE = (6, 5)
# ----------------------

FIG_DIR.mkdir(parents=True, exist_ok=True)
data_year = master[master[YEAR_COL] == FIT_YEAR]

for y_col in Y_COLS:
    fig_file = FIG_DIR / f"{y_col}_vs_{POP_COL}_by_country_{FIT_YEAR}.png"

    if fig_file.exists():
        print("skip, already exists:", fig_file.relative_to(ROOT))
        plt.figure(figsize=FIG_SIZE)
        plt.imshow(plt.imread(fig_file))
        plt.axis("off")
        plt.show()
        continue

    valid = data_year[[ID_COL, COUNTRY_COL, POP_COL, y_col]].dropna()
    valid = valid[(valid[POP_COL] > 0) & (valid[y_col] > 0)]

    # alpha, beta, R2 of the pooled (V4, full sample) fit from the saved table
    r = beta_table[(beta_table["group"] == ALL_GROUP) & (beta_table["sample"] == "full")
                   & (beta_table["y_variable"] == y_col)].iloc[0]

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    for country, color in COUNTRY_COLORS.items():
        sub = valid[valid[COUNTRY_COL] == country]
        is_cap = sub[ID_COL].isin(CAPITAL_IDS)
        ax.scatter(np.log(sub.loc[~is_cap, POP_COL]), np.log(sub.loc[~is_cap, y_col]),
                   color=color, edgecolor="white", s=40, zorder=3, label=country)
        ax.scatter(np.log(sub.loc[is_cap, POP_COL]), np.log(sub.loc[is_cap, y_col]),
                   color=color, edgecolor="black", marker="*", s=220, zorder=4)

    x_line = np.linspace(np.log(valid[POP_COL]).min(), np.log(valid[POP_COL]).max(), 100)
    ax.plot(x_line, r["alpha"] + r["beta"] * x_line, color="tab:red", lw=2, zorder=2)

    ax.text(0.05, 0.92, f"β={r['beta']:.3f}   R²={r['r_squared']:.2f}",
            transform=ax.transAxes, fontsize=12, va="top")
    ax.set_title(f"{y_col} ~ {POP_COL} ({FIT_YEAR})")
    ax.set_xlabel(f"LN[{POP_COL}]")
    ax.set_ylabel(f"LN[{y_col}]")
    ax.legend(title="star = capital", loc="lower right", fontsize=8)

    fig.tight_layout()
    fig.savefig(fig_file, dpi=150, bbox_inches="tight")
    print("saved:", fig_file.relative_to(ROOT))
    plt.show()

In [ ]:
# ---- Config block ----
TS_COLS = ["gdp_total", "employment", "active_enterprises"]  # variables with a yearly value
TS_FILE = OUT_DIR / "beta_by_country_timeseries.csv"
# ----------------------

if TS_FILE.exists():
    print("skip, already exists:", TS_FILE.relative_to(ROOT))
    beta_ts = pd.read_csv(TS_FILE)
else:
    rows = []
    for y_col in TS_COLS:
        # Years with data are taken from the table for each variable
        years = sorted(int(y) for y in master.loc[master[y_col].notna(), YEAR_COL].unique())
        print(y_col, "years with data:", years[0], "-", years[-1])
        for year in years:
            data_y = master[master[YEAR_COL] == year]
            groups = {c: data_y[data_y[COUNTRY_COL] == c]
                      for c in sorted(data_y[COUNTRY_COL].unique())}
            groups[ALL_GROUP] = data_y
            for group, gdf in groups.items():
                samples = {"full": gdf,
                           "no_capitals": gdf[~gdf[ID_COL].isin(CAPITAL_IDS)]}
                for sample, sdf in samples.items():
                    try:
                        row = fit_with_hc3(sdf, y_col, POP_COL)
                    except Exception as e:
                        print("failed:", year, group, sample, y_col, type(e).__name__)
                        continue
                    row.update({"year": year, "group": group, "sample": sample})
                    rows.append(row)
    beta_ts = pd.DataFrame(rows)
    beta_ts.to_csv(TS_FILE, index=False)
    print("saved:", TS_FILE.relative_to(ROOT))

# Compact view: beta by year, full sample
view = beta_ts[beta_ts["sample"] == "full"]
print(view.pivot_table(index=["y_variable", "group"], columns="year", values="beta").round(2).to_string())